# Task E - evaluation analysis (ignore-aware)

Scores checkpoints with `analysis.evaluation.ignore_eval` on the **unmasked** val/test split.
REVIEW_REQUIRED regions (and optionally the PROPOSED sub-threshold regions) are ignored: detections there are neither
TP nor FP. Ultralytics' own val mAP has no ignore support and is shown as a PROVISIONAL reference only.
Labels are automated S2-H boxes, **not human-verified**.

In [ ]:
import sys, json
from pathlib import Path
BACKEND = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
BACKEND = BACKEND if (BACKEND / 'training').exists() else BACKEND.parent
sys.path.insert(0, str(BACKEND))
ROOT = BACKEND.parent
from analysis.evaluation.ignore_eval import evaluate, load_split
from training.evaluate import DS, predict, score

## 1. Sanity check of the evaluator on the real val split (labels as predictions, no model needed)

In [ ]:
import numpy as np
from analysis.evaluation.ignore_eval import ImageEval
base = load_split(DS, 'val', None, [DS / 'manifests/eval_ignore_regions.json'])
fake = {k: ImageEval(v.gt, np.c_[v.gt, np.full(len(v.gt), 0.9)], v.ignore) for k, v in base.items()}
print({k: round(v, 4) if isinstance(v, float) else v for k, v in evaluate(fake).items() if k != 'by_size'})

## 2. Evaluate all runs in the registry (needs trained models)

In [ ]:
reg = ROOT / 'experiments/task-e/runs_registry.jsonl'
runs = [json.loads(l) for l in reg.read_text().splitlines()] if reg.exists() else []
results = {}
for m in runs:
    w = ROOT / 'experiments/task-e/runs' / m['run'] / 'weights/last.pt'
    if w.exists():
        pdir = predict(w, 'val', ROOT / 'experiments/task-e/eval' / m['run'])
        results[m['run']] = score(pdir, 'val', subthreshold=True)
print(len(results), 'runs evaluated')

## 3. A vs B (mean +/- spread over seeds)

In [ ]:
import collections
agg = collections.defaultdict(list)
for run, r in results.items():
    agg[run.rsplit('_s', 1)[0]].append(r)
for arm, rs in sorted(agg.items()):
    ap = [r['AP50'] for r in rs]; small = [r['by_size']['small']['AP50'] for r in rs]
    print(f"{arm:10s} n={len(rs)} AP50 {np.mean(ap):.3f} +/- {np.std(ap):.3f} | small AP50 {np.nanmean(small):.3f}")